In [1]:
#Ejemplos

In [2]:
import numpy as np
from numpy.linalg import eig
from scipy.optimize import linprog, minimize

# =========================================================
# Configuration
# =========================================================

SEED = 42
N_EXAMPLE = 5
MAX_TRIES = 80000
ALPHA_VALUES = list(range(2, 10))
BASELINE_ALPHA = 2
RANK_TOL = 1e-10

# Bounds for optimisation-based methods
LOG_WEIGHT_BOUND = 8.0
OPT_MAXITER = 300

np.random.seed(SEED)


# =========================================================
# Utility functions
# =========================================================

def normalize_positive(w):
    w = np.asarray(w, dtype=float)

    if np.any(~np.isfinite(w)):
        raise ValueError("Priority vector contains non-finite values.")

    if np.any(w < 0):
        raise ValueError("Priority vector contains negative values.")

    total = np.sum(w)

    if total <= 0:
        raise ValueError("Priority vector has non-positive sum.")

    return w / total


def geometric_mean_start(A):
    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    gm = normalize_positive(gm)
    return np.log(gm)


def log_variables_to_weights(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)
    z = z - np.max(z)
    w = np.exp(z)
    return w / np.sum(w)


def fixed_scale_log_vector(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)
    return np.concatenate([z, np.array([0.0])])


def ratio_matrix_from_log_vector(x):
    x = np.asarray(x, dtype=float)
    d = x[:, None] - x[None, :]
    d = np.clip(d, -2 * LOG_WEIGHT_BOUND, 2 * LOG_WEIGHT_BOUND)
    return np.exp(d)


# =========================================================
# Priority derivation methods
# =========================================================

def eigenvector_priority(A):
    eigenvalues, eigenvectors = eig(A)
    idx = np.argmax(eigenvalues.real)

    w = eigenvectors[:, idx].real

    if np.sum(w) < 0:
        w = -w

    if np.any(w <= 0):
        w = np.abs(w)

    return normalize_positive(w)


def geometric_mean_priority(A):
    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    return normalize_positive(gm)


def row_sum_priority(A):
    rs = np.sum(A, axis=1)
    return normalize_positive(rs)


def arithmetic_mean_priority(A):
    # Same ranking as row_sum_priority, but included for completeness.
    am = np.mean(A, axis=1)
    return normalize_positive(am)


def column_sum_priority(A):
    col_sums = np.sum(A, axis=0)
    norm_matrix = A / col_sums
    w = np.sum(norm_matrix, axis=1)
    return normalize_positive(w)


def sscsm_priority(A):
    # Same ranking as column_sum_priority, but included for completeness.
    col_sums = np.sum(A, axis=0)
    w = np.sum(A / col_sums, axis=1)
    return normalize_positive(w)


def harmonic_mean_priority(A):
    n = A.shape[0]
    hm = n / np.sum(1.0 / A, axis=1)
    return normalize_positive(hm)


def cosine_maximization_priority(A):
    """
    Cosine maximisation method:
        max cosine(A, [w_i/w_j])

    Solved in bounded log-weight variables.
    """

    n = A.shape[0]
    norm_A = np.linalg.norm(A)

    if norm_A <= 0:
        raise ValueError("Invalid matrix norm.")

    log_start = geometric_mean_start(A)
    z0 = log_start[:-1] - log_start[-1]
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * (n - 1)

    def objective(z):
        x = fixed_scale_log_vector(z)
        R = ratio_matrix_from_log_vector(x)

        numerator = np.sum(A * R)
        norm_R = np.linalg.norm(R)

        if norm_R <= 0 or not np.isfinite(norm_R):
            return 1e100

        cosine = numerator / (norm_A * norm_R)

        if not np.isfinite(cosine):
            return 1e100

        return -cosine

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"CMM optimisation did not converge: {res.message}")

    x = fixed_scale_log_vector(res.x)
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


def log_chebyshev_priority(A):
    """
    Log-Chebyshev method via linear programming:
        min max_ij |log(a_ij) - (x_i - x_j)|
    """

    n = A.shape[0]
    logA = np.log(A)

    num_vars = n + 1

    c = np.zeros(num_vars)
    c[-1] = 1.0

    bounds = [(None, None)] * n + [(0, None)]

    A_ub = []
    b_ub = []

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            # logA_ij - (x_i - x_j) <= t
            row1 = np.zeros(num_vars)
            row1[i] = -1.0
            row1[j] = 1.0
            row1[-1] = -1.0
            A_ub.append(row1)
            b_ub.append(-logA[i, j])

            # (x_i - x_j) - logA_ij <= t
            row2 = np.zeros(num_vars)
            row2[i] = 1.0
            row2[j] = -1.0
            row2[-1] = -1.0
            A_ub.append(row2)
            b_ub.append(logA[i, j])

    A_ub = np.asarray(A_ub)
    b_ub = np.asarray(b_ub)

    # Fix x_1 = 0 to remove translation invariance
    A_eq = np.zeros((1, num_vars))
    A_eq[0, 0] = 1.0
    b_eq = np.array([0.0])

    res = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not res.success:
        raise RuntimeError("Log-Chebyshev LP did not converge.")

    x = res.x[:n]
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


def least_squares_priority(A):
    """
    Ordinary least squares method:
        min sum_ij (a_ij - w_i/w_j)^2
    """

    n = A.shape[0]

    log_start = geometric_mean_start(A)
    z0 = log_start[:-1] - log_start[-1]
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * (n - 1)

    def objective(z):
        x = fixed_scale_log_vector(z)
        R = ratio_matrix_from_log_vector(x)

        value = np.sum((A - R) ** 2)

        if not np.isfinite(value):
            return 1e100

        return value

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"LSM optimisation did not converge: {res.message}")

    x = fixed_scale_log_vector(res.x)
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


def weighted_least_squares_priority(A):
    """
    Weighted least squares method:
        min sum_ij (a_ij w_j - w_i)^2
    """

    n = A.shape[0]

    z0 = geometric_mean_start(A)
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * n

    def objective(z):
        w = log_variables_to_weights(z)
        residual = A * w[None, :] - w[:, None]

        value = np.sum(residual ** 2)

        if not np.isfinite(value):
            return 1e100

        return value

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"WLSM optimisation did not converge: {res.message}")

    w = log_variables_to_weights(res.x)

    return normalize_positive(w)


def express_ahp_priority(A):
    """
    Reference-alternative / Express AHP style score.

    The reference alternative is chosen as the alternative with largest
    geometric mean score.
    """

    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    best = np.argmax(gm)

    # Scores relative to the selected reference alternative.
    w = A[:, best]

    return normalize_positive(w)


# =========================================================
# Ranking and comparison
# =========================================================

def ranking(weights):
    """
    Return ranking as zero-based indices.
    """
    return np.argsort(-weights)


def ranking_one_based(weights):
    """
    Return ranking as one-based indices, for paper-friendly output.
    """
    return ranking(weights) + 1


def pairwise_order_matrix(weights, tol=RANK_TOL):
    w = np.asarray(weights, dtype=float)
    n = len(w)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    order = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        for j in range(n):
            diff = w[i] - w[j]

            if diff > eps:
                order[i, j] = 1
            elif diff < -eps:
                order[i, j] = -1
            else:
                order[i, j] = 0

    return order


def same_ranking(w1, w2, tol=RANK_TOL):
    return np.array_equal(
        pairwise_order_matrix(w1, tol),
        pairwise_order_matrix(w2, tol)
    )


# =========================================================
# Ordinal PCM generator
# =========================================================

def random_ordinal_pcm(n):
    """
    Generate random ordinal reciprocal matrix with entries:
    {1, P, 1/P}.
    """

    A = np.ones((n, n), dtype=object)

    for i in range(n):
        for j in range(i + 1, n):

            choice = np.random.choice(["tie", "pref", "notpref"])

            if choice == "tie":
                A[i, j] = 1
                A[j, i] = 1

            elif choice == "pref":
                A[i, j] = "P"
                A[j, i] = "1/P"

            else:
                A[i, j] = "1/P"
                A[j, i] = "P"

    return A


def substitute_alpha(A_ord, alpha):
    n = A_ord.shape[0]
    A = np.ones((n, n), dtype=float)

    for i in range(n):
        for j in range(n):

            if A_ord[i, j] == "P":
                A[i, j] = alpha
            elif A_ord[i, j] == "1/P":
                A[i, j] = 1.0 / alpha
            else:
                A[i, j] = 1.0

    return A


# =========================================================
# Pretty printing
# =========================================================

def print_matrix(A):
    for row in A:
        print("   ".join(f"{x:7.3f}" for x in row))


def print_weights(w):
    print(", ".join(f"{x:.4f}" for x in w))


def print_latex_matrix_from_ordinal(A_ord):
    """
    Print a LaTeX matrix in terms of alpha.
    """

    n = A_ord.shape[0]

    print("\\[")
    print("A(\\alpha)=")
    print("\\begin{bmatrix}")

    for i in range(n):
        row = []

        for j in range(n):
            val = A_ord[i, j]

            if val == "P":
                row.append("\\alpha")
            elif val == "1/P":
                row.append("1/\\alpha")
            else:
                row.append("1")

        line = " & ".join(row)

        if i < n - 1:
            line += " \\\\"

        print(line)

    print("\\end{bmatrix}.")
    print("\\]")


# =========================================================
# Find and display explicit rank reversal
# =========================================================

def find_and_print_example(method, method_name, n=N_EXAMPLE, max_tries=MAX_TRIES):
    """
    Search for one explicit IOP rank reversal example.
    """

    for attempt in range(1, max_tries + 1):

        A_ord = random_ordinal_pcm(n)

        A_base = substitute_alpha(A_ord, BASELINE_ALPHA)
        w_base = method(A_base)

        for alpha in ALPHA_VALUES[1:]:

            A_alpha = substitute_alpha(A_ord, alpha)
            w_alpha = method(A_alpha)

            if not same_ranking(w_base, w_alpha):

                print("=" * 70)
                print(f"METHOD: {method_name}")
                print(f"Matrix size n = {n}")
                print(f"Attempt = {attempt}")
                print("-" * 70)

                print("\nSymbolic matrix:")
                print_latex_matrix_from_ordinal(A_ord)

                print(f"\nBaseline alpha = {BASELINE_ALPHA}")
                print(f"A({BASELINE_ALPHA}):")
                print_matrix(A_base)

                print("\nPriority vector:")
                print_weights(w_base)

                print("Ranking:", ranking_one_based(w_base))

                print(f"\nIntensified alpha = {alpha}")
                print(f"A({alpha}):")
                print_matrix(A_alpha)

                print("\nPriority vector:")
                print_weights(w_alpha)

                print("Ranking:", ranking_one_based(w_alpha))

                print("\n>>> IOP RANK REVERSAL DETECTED <<<")
                print("=" * 70 + "\n")

                return A_ord, BASELINE_ALPHA, alpha, w_base, w_alpha

    print(f"No example found for {method_name} after {max_tries} attempts.\n")
    return None


# =========================================================
# Main
# =========================================================

methods_for_examples = {
    # In the order of Section 2.2, excluding GMM because it is invariant.
    "Eigenvector method": eigenvector_priority,
    "Row sum method": row_sum_priority,
    "Column sum method": column_sum_priority,
    "Harmonic mean method": harmonic_mean_priority,
    "Cosine maximisation method": cosine_maximization_priority,
    "Log-Chebyshev method": log_chebyshev_priority,
    "Least squares method": least_squares_priority,
    "Weighted least squares method": weighted_least_squares_priority,

    # Optional additional methods from your earlier code:
    # "Arithmetic mean method": arithmetic_mean_priority,
    # "SSCSM": sscsm_priority,
    # "Express AHP": express_ahp_priority,
}

print("\nSearching explicit IOP rank reversal examples...\n")

examples_found = {}

for name, method in methods_for_examples.items():
    try:
        result = find_and_print_example(method, name, n=N_EXAMPLE, max_tries=MAX_TRIES)
        examples_found[name] = result
    except Exception as exc:
        print("=" * 70)
        print(f"METHOD: {name}")
        print(f"An error occurred: {exc}")
        print("=" * 70 + "\n")


# =========================================================
# Check geometric mean invariance
# =========================================================

print("\nChecking invariance of the geometric mean method...\n")

for test_id in range(5):
    A_ord = random_ordinal_pcm(N_EXAMPLE)

    A2 = substitute_alpha(A_ord, 2)
    A9 = substitute_alpha(A_ord, 9)

    w2 = geometric_mean_priority(A2)
    w9 = geometric_mean_priority(A9)

    print(f"Test {test_id + 1}")
    print("Ranking at alpha=2:", ranking_one_based(w2))
    print("Ranking at alpha=9:", ranking_one_based(w9))
    print("Same ranking:", same_ranking(w2, w9))
    print()

print("Geometric mean ranking is invariant under uniform preference intensification.\n")


Searching explicit IOP rank reversal examples...

METHOD: Eigenvector method
Matrix size n = 5
Attempt = 3
----------------------------------------------------------------------

Symbolic matrix:
\[
A(\alpha)=
\begin{bmatrix}
1 & 1 & 1 & \alpha & \alpha \\
1 & 1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1/\alpha & 1/\alpha \\
1/\alpha & 1 & \alpha & 1 & 1/\alpha \\
1/\alpha & 1 & \alpha & \alpha & 1
\end{bmatrix}.
\]

Baseline alpha = 2
A(2):
  1.000     1.000     1.000     2.000     2.000
  1.000     1.000     1.000     1.000     1.000
  1.000     1.000     1.000     0.500     0.500
  0.500     1.000     2.000     1.000     0.500
  0.500     1.000     2.000     2.000     1.000

Priority vector:
0.2636, 0.1888, 0.1514, 0.1713, 0.2249
Ranking: [1 5 2 4 3]

Intensified alpha = 9
A(9):
  1.000     1.000     1.000     9.000     9.000
  1.000     1.000     1.000     1.000     1.000
  1.000     1.000     1.000     0.111     0.111
  0.111     1.000     9.000     1.000     0.111
  0.111     1.000     9.000 

In [3]:
#Ejemplos donde cambia el primer término

In [6]:
# =========================================================
# Explicit top-ranked alternative change examples
# =========================================================

import os
import numpy as np
from numpy.linalg import eig
from scipy.optimize import linprog, minimize

# =========================================================
# Configuration
# =========================================================

SEED = 42

BASELINE_ALPHA = 2
ALPHA_VALUES = list(range(9, 2, -1))   # Try alpha = 9,8,...,3 first
RANK_TOL = 1e-10

N_EXAMPLE = 7
MAX_TRIES = 1000000
PROGRESS_EVERY = 50000

LOG_WEIGHT_BOUND = 8.0
OPT_MAXITER = 300

OUTPUT_DIR = "top_rank_examples"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(SEED)


# =========================================================
# Utility functions
# =========================================================

def normalize_positive(w):
    w = np.asarray(w, dtype=float)

    if np.any(~np.isfinite(w)):
        raise ValueError("Priority vector contains non-finite values.")

    if np.any(w < 0):
        raise ValueError("Priority vector contains negative values.")

    total = np.sum(w)

    if total <= 0:
        raise ValueError("Priority vector has non-positive sum.")

    return w / total


def geometric_mean_start(A):
    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    gm = normalize_positive(gm)
    return np.log(gm)


def log_variables_to_weights(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)
    z = z - np.max(z)
    w = np.exp(z)
    return w / np.sum(w)


def fixed_scale_log_vector(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)
    return np.concatenate([z, np.array([0.0])])


def ratio_matrix_from_log_vector(x):
    x = np.asarray(x, dtype=float)
    d = x[:, None] - x[None, :]
    d = np.clip(d, -2 * LOG_WEIGHT_BOUND, 2 * LOG_WEIGHT_BOUND)
    return np.exp(d)


# =========================================================
# Priority derivation methods
# =========================================================

def eigenvector_priority(A):
    eigenvalues, eigenvectors = eig(A)
    idx = np.argmax(eigenvalues.real)

    w = eigenvectors[:, idx].real

    if np.sum(w) < 0:
        w = -w

    if np.any(w <= 0):
        w = np.abs(w)

    return normalize_positive(w)


def geometric_mean_priority(A):
    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    return normalize_positive(gm)


def row_sum_priority(A):
    rs = np.sum(A, axis=1)
    return normalize_positive(rs)


def column_sum_priority(A):
    col_sums = np.sum(A, axis=0)
    norm_matrix = A / col_sums
    w = np.sum(norm_matrix, axis=1)
    return normalize_positive(w)


def harmonic_mean_priority(A):
    n = A.shape[0]
    hm = n / np.sum(1.0 / A, axis=1)
    return normalize_positive(hm)


def cosine_maximization_priority(A):
    n = A.shape[0]
    norm_A = np.linalg.norm(A)

    if norm_A <= 0:
        raise ValueError("Invalid matrix norm.")

    log_start = geometric_mean_start(A)
    z0 = log_start[:-1] - log_start[-1]
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * (n - 1)

    def objective(z):
        x = fixed_scale_log_vector(z)
        R = ratio_matrix_from_log_vector(x)

        numerator = np.sum(A * R)
        norm_R = np.linalg.norm(R)

        if norm_R <= 0 or not np.isfinite(norm_R):
            return 1e100

        cosine = numerator / (norm_A * norm_R)

        if not np.isfinite(cosine):
            return 1e100

        return -cosine

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"CMM optimisation did not converge: {res.message}")

    x = fixed_scale_log_vector(res.x)
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


def log_chebyshev_priority(A):
    n = A.shape[0]
    logA = np.log(A)

    num_vars = n + 1

    c = np.zeros(num_vars)
    c[-1] = 1.0

    bounds = [(None, None)] * n + [(0, None)]

    A_ub = []
    b_ub = []

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            row1 = np.zeros(num_vars)
            row1[i] = -1.0
            row1[j] = 1.0
            row1[-1] = -1.0
            A_ub.append(row1)
            b_ub.append(-logA[i, j])

            row2 = np.zeros(num_vars)
            row2[i] = 1.0
            row2[j] = -1.0
            row2[-1] = -1.0
            A_ub.append(row2)
            b_ub.append(logA[i, j])

    A_ub = np.asarray(A_ub)
    b_ub = np.asarray(b_ub)

    # Fix x_1 = 0
    A_eq = np.zeros((1, num_vars))
    A_eq[0, 0] = 1.0
    b_eq = np.array([0.0])

    res = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not res.success:
        raise RuntimeError("Log-Chebyshev LP did not converge.")

    x = res.x[:n]
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


def least_squares_priority(A):
    n = A.shape[0]

    log_start = geometric_mean_start(A)
    z0 = log_start[:-1] - log_start[-1]
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * (n - 1)

    def objective(z):
        x = fixed_scale_log_vector(z)
        R = ratio_matrix_from_log_vector(x)

        value = np.sum((A - R) ** 2)

        if not np.isfinite(value):
            return 1e100

        return value

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"LSM optimisation did not converge: {res.message}")

    x = fixed_scale_log_vector(res.x)
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


def weighted_least_squares_priority(A):
    n = A.shape[0]

    z0 = geometric_mean_start(A)
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * n

    def objective(z):
        w = log_variables_to_weights(z)
        residual = A * w[None, :] - w[:, None]

        value = np.sum(residual ** 2)

        if not np.isfinite(value):
            return 1e100

        return value

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"WLSM optimisation did not converge: {res.message}")

    w = log_variables_to_weights(res.x)

    return normalize_positive(w)


# =========================================================
# Ranking and top-ranked alternative comparison
# =========================================================

def ranking(weights):
    return np.argsort(-weights)


def ranking_one_based(weights):
    return ranking(weights) + 1


def top_set(weights, tol=RANK_TOL):
    w = np.asarray(weights, dtype=float)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    max_w = np.max(w)

    return set(np.where(max_w - w <= eps)[0])


def top_rank_change_occurred(w_base, w_new, tol=RANK_TOL):
    base_top = top_set(w_base, tol)
    new_top = top_set(w_new, tol)

    return len(base_top.intersection(new_top)) == 0


# =========================================================
# Random ordinal PCM generator
# =========================================================

def random_ordinal_pcm(n):
    A = np.ones((n, n), dtype=object)

    for i in range(n):
        for j in range(i + 1, n):

            choice = np.random.choice(["tie", "pref", "notpref"])

            if choice == "tie":
                A[i, j] = 1
                A[j, i] = 1

            elif choice == "pref":
                A[i, j] = "P"
                A[j, i] = "1/P"

            else:
                A[i, j] = "1/P"
                A[j, i] = "P"

    return A


def substitute_alpha(A_ord, alpha):
    n = A_ord.shape[0]
    A = np.ones((n, n), dtype=float)

    for i in range(n):
        for j in range(n):

            if A_ord[i, j] == "P":
                A[i, j] = alpha
            elif A_ord[i, j] == "1/P":
                A[i, j] = 1.0 / alpha
            else:
                A[i, j] = 1.0

    return A


# =========================================================
# Pretty printing and LaTeX output
# =========================================================

def print_matrix(A):
    for row in A:
        print("   ".join(f"{x:7.3f}" for x in row))


def print_weights(w):
    print(", ".join(f"{x:.4f}" for x in w))


def latex_matrix_from_ordinal(A_ord):
    n = A_ord.shape[0]
    lines = []

    lines.append("\\[")
    lines.append("A(\\alpha)=")
    lines.append("\\begin{bmatrix}")

    for i in range(n):
        row = []

        for j in range(n):
            val = A_ord[i, j]

            if val == "P":
                row.append("\\alpha")
            elif val == "1/P":
                row.append("1/\\alpha")
            else:
                row.append("1")

        line = " & ".join(row)

        if i < n - 1:
            line += " \\\\"

        lines.append(line)

    lines.append("\\end{bmatrix}.")
    lines.append("\\]")

    return "\n".join(lines)


def latex_vector(w, decimals=3):
    return "(" + ",\\; ".join(f"{x:.{decimals}f}" for x in w) + ")"


def latex_ranking(rank):
    return " \\succ ".join(str(int(x)) for x in rank)


# =========================================================
# Find top-ranked alternative change example
# =========================================================

def find_top_change_example(
    method,
    method_name,
    n=N_EXAMPLE,
    max_tries=MAX_TRIES,
    progress_every=PROGRESS_EVERY
):
    for attempt in range(1, max_tries + 1):

        if attempt % progress_every == 0:
            print(f"  {method_name}: {attempt} attempts checked...")

        A_ord = random_ordinal_pcm(n)

        try:
            A_base = substitute_alpha(A_ord, BASELINE_ALPHA)
            w_base = method(A_base)
        except Exception:
            continue

        top_base = top_set(w_base)

        for alpha in ALPHA_VALUES:

            try:
                A_alpha = substitute_alpha(A_ord, alpha)
                w_alpha = method(A_alpha)
            except Exception:
                continue

            top_alpha = top_set(w_alpha)

            if top_rank_change_occurred(w_base, w_alpha):

                result = {
                    "method": method_name,
                    "n": n,
                    "attempt": attempt,
                    "baseline_alpha": BASELINE_ALPHA,
                    "intensified_alpha": alpha,
                    "A_ord": A_ord,
                    "A_base": A_base,
                    "A_alpha": A_alpha,
                    "w_base": w_base,
                    "w_alpha": w_alpha,
                    "ranking_base": ranking_one_based(w_base),
                    "ranking_alpha": ranking_one_based(w_alpha),
                    "top_base": sorted([i + 1 for i in top_base]),
                    "top_alpha": sorted([i + 1 for i in top_alpha]),
                }

                return result

    return None


# =========================================================
# Display and save example
# =========================================================

def display_example(result):
    if result is None:
        return

    print("=" * 70)
    print(f"METHOD: {result['method']}")
    print(f"Matrix size n = {result['n']}")
    print(f"Attempt = {result['attempt']}")
    print("-" * 70)

    print("\nSymbolic matrix:")
    print(latex_matrix_from_ordinal(result["A_ord"]))

    print(f"\nBaseline alpha = {result['baseline_alpha']}")
    print(f"A({result['baseline_alpha']}):")
    print_matrix(result["A_base"])

    print("\nPriority vector:")
    print_weights(result["w_base"])

    print("Ranking:", result["ranking_base"])
    print("Top-ranked alternative(s):", result["top_base"])

    print(f"\nIntensified alpha = {result['intensified_alpha']}")
    print(f"A({result['intensified_alpha']}):")
    print_matrix(result["A_alpha"])

    print("\nPriority vector:")
    print_weights(result["w_alpha"])

    print("Ranking:", result["ranking_alpha"])
    print("Top-ranked alternative(s):", result["top_alpha"])

    print("\n>>> TOP-RANKED ALTERNATIVE CHANGE DETECTED <<<")
    print("=" * 70 + "\n")


def save_example_as_latex(result):
    if result is None:
        return

    safe_name = (
        result["method"]
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    file_path = os.path.join(
        OUTPUT_DIR,
        f"top_change_example_{safe_name}.tex"
    )

    alpha0 = result["baseline_alpha"]
    alpha1 = result["intensified_alpha"]

    text = []

    text.append(f"% Top-ranked alternative change example for {result['method']}")
    text.append("")
    text.append(latex_matrix_from_ordinal(result["A_ord"]))
    text.append("")
    text.append(f"For $\\alpha={alpha0}$, the priority vector is")
    text.append("\\[")
    text.append(f"w_{{A({alpha0})}}={latex_vector(result['w_base'])},")
    text.append("\\]")
    text.append("which induces the ranking")
    text.append("\\[")
    text.append(latex_ranking(result["ranking_base"]) + ".")
    text.append("\\]")
    text.append("")
    text.append(
        f"After uniform intensification to $\\alpha={alpha1}$, "
        "the priority vector becomes"
    )
    text.append("\\[")
    text.append(f"w_{{A({alpha1})}}={latex_vector(result['w_alpha'])},")
    text.append("\\]")
    text.append("leading to")
    text.append("\\[")
    text.append(latex_ranking(result["ranking_alpha"]) + ".")
    text.append("\\]")
    text.append("")
    text.append(
        "Thus, the top-ranked alternative changes under uniform preference intensification."
    )

    with open(file_path, "w", encoding="utf-8") as f:
        f.write("\n".join(text))

    print(f"LaTeX example saved to: {file_path}")


# =========================================================
# Methods and search settings
# =========================================================

methods_for_top_examples = {
    "Eigenvector method": eigenvector_priority,
    "Row sum method": row_sum_priority,
    "Column sum method": column_sum_priority,
    "Harmonic mean method": harmonic_mean_priority,
    "Cosine maximisation method": cosine_maximization_priority,
    "Log-Chebyshev method": log_chebyshev_priority,
    "Least squares method": least_squares_priority,
    "Weighted least squares method": weighted_least_squares_priority,
}

search_settings = {
    "Eigenvector method": {
        "n": 7,
        "max_tries": 500000,
    },
    "Row sum method": {
        "n": 8,
        "max_tries": 2000000,
    },
    "Column sum method": {
        "n": 7,
        "max_tries": 1000000,
    },
    "Harmonic mean method": {
        "n": 7,
        "max_tries": 1000000,
    },
    "Cosine maximisation method": {
        "n": 6,
        "max_tries": 300000,
    },
    "Log-Chebyshev method": {
        "n": 6,
        "max_tries": 300000,
    },
    "Least squares method": {
        "n": 6,
        "max_tries": 300000,
    },
    "Weighted least squares method": {
        "n": 6,
        "max_tries": 300000,
    },
}


# =========================================================
# Main search
# =========================================================

print("\nSearching top-ranked alternative change examples...\n")

top_examples_found = {}

for name, method in methods_for_top_examples.items():

    settings = search_settings.get(
        name,
        {
            "n": N_EXAMPLE,
            "max_tries": MAX_TRIES,
        }
    )

    print("=" * 70)
    print(f"Searching: {name}")
    print(f"n = {settings['n']}")
    print(f"max tries = {settings['max_tries']}")
    print("=" * 70)

    result = find_top_change_example(
        method=method,
        method_name=name,
        n=settings["n"],
        max_tries=settings["max_tries"],
        progress_every=PROGRESS_EVERY
    )

    top_examples_found[name] = result

    if result is None:
        print(
            f"No top-ranked alternative change found for {name} "
            f"after {settings['max_tries']} attempts.\n"
        )
    else:
        display_example(result)
        save_example_as_latex(result)


# =========================================================
# Check geometric mean invariance
# =========================================================

print("\nChecking invariance of the geometric mean method...\n")

for test_id in range(5):

    A_ord = random_ordinal_pcm(N_EXAMPLE)

    A2 = substitute_alpha(A_ord, 2)
    A9 = substitute_alpha(A_ord, 9)

    w2 = geometric_mean_priority(A2)
    w9 = geometric_mean_priority(A9)

    print(f"Test {test_id + 1}")
    print("Ranking at alpha=2:", ranking_one_based(w2))
    print("Ranking at alpha=9:", ranking_one_based(w9))
    print("Top set at alpha=2:", sorted([i + 1 for i in top_set(w2)]))
    print("Top set at alpha=9:", sorted([i + 1 for i in top_set(w9)]))
    print("Top-ranked alternative changes:", top_rank_change_occurred(w2, w9))
    print()

print("Geometric mean method is invariant under uniform preference intensification.\n")


Searching top-ranked alternative change examples...

Searching: Eigenvector method
n = 7
max tries = 500000
METHOD: Eigenvector method
Matrix size n = 7
Attempt = 5
----------------------------------------------------------------------

Symbolic matrix:
\[
A(\alpha)=
\begin{bmatrix}
1 & 1 & \alpha & 1/\alpha & 1 & \alpha & 1 \\
1 & 1 & 1 & 1 & 1 & 1/\alpha & 1 \\
1/\alpha & 1 & 1 & 1 & 1 & 1/\alpha & 1 \\
\alpha & 1 & 1 & 1 & 1 & 1/\alpha & 1/\alpha \\
1 & 1 & 1 & 1 & 1 & 1/\alpha & 1 \\
1/\alpha & \alpha & \alpha & \alpha & \alpha & 1 & 1/\alpha \\
1 & 1 & 1 & \alpha & 1 & \alpha & 1
\end{bmatrix}.
\]

Baseline alpha = 2
A(2):
  1.000     1.000     2.000     0.500     1.000     2.000     1.000
  1.000     1.000     1.000     1.000     1.000     0.500     1.000
  0.500     1.000     1.000     1.000     1.000     0.500     1.000
  2.000     1.000     1.000     1.000     1.000     0.500     0.500
  1.000     1.000     1.000     1.000     1.000     0.500     1.000
  0.500     2.000     2